In [17]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
import os

#Load the secrets
load_dotenv("secret.env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
MINIO_BUCKET_BRONZE = os.getenv("MINIO_BUCKET_BRONZE")

spark = (
    SparkSession.builder
    .appName("Kafka-MinIO-Streaming")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,"
        "org.apache.hadoop:hadoop-aws:3.3.2"
    )
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)

Spark: 3.3.0


In [18]:
df_kafka = (
    spark.readStream
         .format("kafka")
         .option(
             "kafka.bootstrap.servers",
             "kafka:29092"
         )
         .option(
             "subscribe",
             "transactions_stream"
         )
         .option(
             "startingOffsets",
             "earliest"
         )
         .option(
             "maxOffsetsPerTrigger",
             5000
         )
         .load()
)

In [19]:
df_kafka.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [20]:
df_raw_kafka = (
    df_kafka
    .select(
        col("key")
            .cast("string")
            .alias("message_key"),

        col("value")
            .cast("string")
            .alias("json_value"),

        col("topic"),

        col("partition"),

        col("offset"),

        col("timestamp")
            .alias("kafka_timestamp")
    )
)

In [21]:
df_raw_kafka.printSchema()

root
 |-- message_key: string (nullable = true)
 |-- json_value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



In [14]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
    DateType
)

transaction_schema = StructType([
    StructField("step", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("Account_ID", StringType(), True),
    StructField("old_balance", DoubleType(), True),
    StructField("new_balance", DoubleType(), True),
    StructField("receiver_ID", StringType(), True),
    StructField("receiver_old_bal", DoubleType(), True),
    StructField("receiver_new_bal", DoubleType(), True),
    StructField("is_fraud", IntegerType(), True),

    # JSON contains this as "2026-01-21"
    StructField("load_date", StringType(), True)
])

In [22]:
df_parsed = (
    df_raw_kafka

    .withColumn(
        "transaction",
        from_json(
            col("json_value"),
            transaction_schema
        )
    )

    .select(
        "message_key",
        "transaction.*",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp"
    )

    .withColumn(
        "load_date",
        to_date(
            col("load_date"),
            "yyyy-MM-dd"
        )
    )
)

In [23]:
df_parsed.printSchema()

root
 |-- message_key: string (nullable = true)
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- Account_ID: string (nullable = true)
 |-- old_balance: double (nullable = true)
 |-- new_balance: double (nullable = true)
 |-- receiver_ID: string (nullable = true)
 |-- receiver_old_bal: double (nullable = true)
 |-- receiver_new_bal: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- load_date: date (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



In [25]:
from pyspark.sql.functions import (
    col,
    from_json,
    to_date,
    current_timestamp,
    lit
)

df_bronze_stream = (
    df_parsed

    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )

    .withColumn(
        "source",
        lit("kafka")
    )
)

In [26]:
df_bronze_stream.printSchema()

root
 |-- message_key: string (nullable = true)
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- Account_ID: string (nullable = true)
 |-- old_balance: double (nullable = true)
 |-- new_balance: double (nullable = true)
 |-- receiver_ID: string (nullable = true)
 |-- receiver_old_bal: double (nullable = true)
 |-- receiver_new_bal: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- load_date: date (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- source: string (nullable = false)



In [27]:
bronze_path = (
    f"s3a://{MINIO_BUCKET_BRONZE}/transactions_stream/"
)

checkpoint_path = (
    f"s3a://{MINIO_BUCKET_BRONZE}/checkpoints/transactions_stream_v1/"
)

In [28]:
bronze_query = (
    df_bronze_stream
    .writeStream

    .format("parquet")

    .outputMode("append")

    .option(
        "path",
        bronze_path
    )

    .option(
        "checkpointLocation",
        checkpoint_path
    )

    .partitionBy(
        "load_date"
    )

    .trigger(
        processingTime="10 seconds"
    )

    .start()
)

In [29]:
bronze_query.isActive

True

In [30]:
bronze_query.status

{'message': 'Waiting for next trigger',
 'isDataAvailable': True,
 'isTriggerActive': False}

In [31]:
bronze_query.lastProgress

{'id': 'e7458a8c-dbb5-45ce-a808-f8a8e9c8bac0',
 'runId': 'a788ad85-fedf-4725-a67c-c5d602039565',
 'name': None,
 'timestamp': '2026-09-26T17:55:40.001Z',
 'batchId': 7,
 'numInputRows': 5000,
 'inputRowsPerSecond': 500.0,
 'processedRowsPerSecond': 5973.715651135006,
 'durationMs': {'addBatch': 584,
  'getBatch': 1,
  'latestOffset': 3,
  'queryPlanning': 19,
  'triggerExecution': 837,
  'walCommit': 97},
 'stateOperators': [],
 'sources': [{'description': 'KafkaV2[Subscribe[transactions_stream]]',
   'startOffset': {'transactions_stream': {'0': 35000}},
   'endOffset': {'transactions_stream': {'0': 40000}},
   'latestOffset': {'transactions_stream': {'0': 323162}},
   'numInputRows': 5000,
   'inputRowsPerSecond': 500.0,
   'processedRowsPerSecond': 5973.715651135006,
   'metrics': {'avgOffsetsBehindLatest': '283162.0',
    'maxOffsetsBehindLatest': '283162',
    'minOffsetsBehindLatest': '283162'}}],
 'sink': {'description': 'FileSink[s3a://bronzedata/transactions_stream/]',
  'numOu

In [37]:
for progress in bronze_query.recentProgress:
    print(
        "Batch:",
        progress["batchId"],
        "| Rows:",
        progress["numInputRows"],
        "| Input rows/sec:",
        progress["inputRowsPerSecond"],
        "| Processed rows/sec:",
        progress["processedRowsPerSecond"]
    )

Batch: 0 | Rows: 5000 | Input rows/sec: 0.0 | Processed rows/sec: 698.2265046781176
Batch: 1 | Rows: 5000 | Input rows/sec: 697.8367062107467 | Processed rows/sec: 3646.9730123997083
Batch: 2 | Rows: 5000 | Input rows/sec: 526.3711969681019 | Processed rows/sec: 3046.9226081657525
Batch: 3 | Rows: 5000 | Input rows/sec: 500.05000500050005 | Processed rows/sec: 4646.840148698884
Batch: 4 | Rows: 5000 | Input rows/sec: 500.0 | Processed rows/sec: 4582.9514207149405
Batch: 5 | Rows: 5000 | Input rows/sec: 499.9500049995001 | Processed rows/sec: 4672.8971962616815
Batch: 6 | Rows: 5000 | Input rows/sec: 500.0 | Processed rows/sec: 6605.019815059445
Batch: 7 | Rows: 5000 | Input rows/sec: 500.0 | Processed rows/sec: 5973.715651135006
Batch: 8 | Rows: 5000 | Input rows/sec: 500.0 | Processed rows/sec: 6002.400960384154
Batch: 9 | Rows: 5000 | Input rows/sec: 500.0 | Processed rows/sec: 5428.881650380022
Batch: 10 | Rows: 5000 | Input rows/sec: 500.0 | Processed rows/sec: 5668.934240362812
Ba

In [38]:
bronze_query.stop()

In [ ]:
bronze_query.isActive